# VoxCPM2 · Tagalog GPU Quality Assurance

Run this notebook in **Google Colab with a GPU runtime** (Runtime → Change runtime type → GPU). It generates a real Tagalog WAV with VoxCPM2 and mixes it into a short, synthetic test video. No private recordings, paid API key or third-party voice cloning is required. **GPU time and Colab quotas may apply.**

Use the produced artifacts to review pronunciation, prosody, loudness, audio synchronization, and actual model compatibility. The PR's CPU-only CI cannot establish Tagalog voice quality.

In [ ]:
import sys, subprocess, pathlib, json, os
import torch
assert sys.version_info[:2] >= (3, 10) and sys.version_info[:2] < (3, 13), f'Python 3.10-3.12 required; got {sys.version}'
assert torch.cuda.is_available(), 'Choose a GPU runtime in Colab before running this notebook.'
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| total VRAM:', round(gpu.total_memory / (1024**3), 2), 'GiB')
if gpu.total_memory / 1024**3 < 8:
    print('Warning: less than ~8 GiB GPU VRAM; VoxCPM2 might not fit.')


In [ ]:
ROOT = pathlib.Path('/content/video-ai-editor')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feat/voxcpm2-voiceover','https://github.com/urfavxbf/video-ai-editor.git',str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','voxcpm','soundfile'],check=True)
subprocess.run(['ffmpeg','-version'],check=True,stdout=subprocess.DEVNULL)
print('Prepared VoxCPM2 Python adapter and FFmpeg toolchain')


In [ ]:
import time
out = pathlib.Path('/content/voxcpm2-qa')
out.mkdir(parents=True,exist_ok=True)
script = 'Bakit nga ba tayo kinikilabutan kapag nakakarinig ng paborito nating kanta? Alamin natin.'
style = 'Natural Filipino female storyteller, expressive and clear Tagalog pronunciation, warm friendly tone'
started = time.monotonic()
subprocess.run([sys.executable,str(ROOT/'scripts/voxcpm_voiceover.py'),
    '--text',script,'--style',style,'--device','cuda',
    '--output',str(out/'tagalog.wav')],check=True,timeout=600)
print('Synthesis elapsed seconds:',round(time.monotonic()-started,1))
import soundfile as sf
audio,sample_rate = sf.read(str(out/'tagalog.wav'))
assert sample_rate == 48000, f'Unexpected VoxCPM2 output sample rate: {sample_rate}'
assert len(audio) > sample_rate, 'Generated audio is suspiciously short'
print('Voiceover duration:',round(len(audio)/sample_rate,2),'seconds')
from IPython.display import Audio,display
display(Audio(filename=str(out/'tagalog.wav')))


In [ ]:
duration = max(15, min(40, int(len(audio) / sample_rate + 2)))
video = out/'synthetic-test.mp4'
mixed = out/'synthetic-test-voiced.mp4'
subprocess.run(['ffmpeg','-nostdin','-y','-hide_banner','-loglevel','error',
    '-f','lavfi','-i',f'testsrc=size=360x640:rate=24:duration={duration}',
    '-f','lavfi','-i',f'anullsrc=r=16000:cl=mono:d={duration}',
    '-map','0:v:0','-map','1:a:0','-c:v','mpeg4','-q:v','8',
    '-c:a','aac','-t',str(duration),str(video)],check=True)
sys.path.insert(0, str(ROOT/'scripts'))
from voxcpm_voiceover import mux_video
mux_video(video, out/'tagalog.wav', mixed, 0.0)
assert mixed.exists() and mixed.stat().st_size > 2048
print('WAV:',out/'tagalog.wav')
print('MP4:',mixed)
from IPython.display import Video
display(Video(str(mixed),embed=True))
print('Review: Tagalog pronunciation, hallucinated words, prosody, artifact/clipping, and MP4 audio sync.')


## Acceptance checklist

- [ ] Words sound natural to a native Filipino listener
- [ ] No lost or hallucinated words
- [ ] Clear pauses and expressive but realistic pacing
- [ ] 48 kHz valid WAV
- [ ] MP4 has AAC narration and the video length is retained
- [ ] Time and GPU usage are within budget

Report any issues in [GitHub QA Issue #3](https://github.com/urfavxbf/video-ai-editor/issues/3). Never upload private voice references without consent.